In [1]:
from dotenv import load_dotenv
load_dotenv()

from tenantmate.retrieve import search_bm25

for q in ["Tribunal orders", "rental bond", "domestic violence termination", "pet"]:
    print(f"\n=== {q} ===")
    for r in search_bm25(q, k=3):
        print(f"  [{r['score']:.3f}] s{r['section_number']} — {r['section_title']}")

/Users/varunchandrashekar/Tenantmate/code/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/varunchandrashekar/Tenantmate/code/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



=== Tribunal orders ===
  [0.294] s217 — Disputes about listings
  [0.234] s65 — Tenants’ remedies for repairs—Tribunal orders
  [0.200] s47 — Tenant’s remedies for repayment of rent and excess charges

=== rental bond ===
  [1.522] s186A — Regulations may establish rental bond roll-over scheme
  [1.386] s156D — Payment of rental bond during social housing tenancy agreements
  [1.164] s157A — Online rental bond service

=== domestic violence termination ===
  [0.674] s105D — Effect of giving domestic violence termination notice
  [0.326] s105A — Definitions
  [0.315] s105C — Domestic violence termination notice

=== pet ===
  [0.100] s35 — Pets


In [2]:
from tenantmate.retrieve import search_hybrid

for q in ["How much notice for a rent increase?", "Can I keep a pet?", "How do I get my bond back?"]:
    print(f"\n=== {q} ===")
    for r in search_hybrid(q, k=3):
        print(f"  [rrf={r['rrf_score']:.4f}] s{r['section_number']} — {r['section_title']}")


=== How much notice for a rent increase? ===
  [rrf=0.0164] s41 — Rent increases
  [rrf=0.0161] s99 — Rent increases during long-term fixed term leases—termination notice by tenant
  [rrf=0.0159] s44 — Tenant’s remedies for excessive rent

=== Can I keep a pet? ===
  [rrf=0.0311] s35 — Pets
  [rrf=0.0164] s73B — Keeping of pets with landlord’s consent
  [rrf=0.0161] s73E — Reasonable conditions of consent

=== How do I get my bond back? ===
  [rrf=0.0164] s186A — Regulations may establish rental bond roll-over scheme
  [rrf=0.0161] s185 — Rental Bond Account
  [rrf=0.0159] s159 — Payment of bonds


In [1]:
from dotenv import load_dotenv
load_dotenv()

from tenantmate.retrieve import search_full, search_hybrid_rewritten

q = "How do I get my rental bond back at the end of my tenancy?"

print("=== Hybrid + rewrite (no rerank) ===")
for r in search_hybrid_rewritten(q, k=5):
    print(f"  s{r['section_number']} — {r['section_title']}")

print("\n=== Full pipeline (rewrite + hybrid + rerank) ===")
for r in search_full(q, k=5):
    score = r.get("rerank_score", 0)
    print(f"  [{score:+.3f}] s{r['section_number']} — {r['section_title']}")

/Users/varunchandrashekar/Tenantmate/code/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/varunchandrashekar/Tenantmate/code/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== Hybrid + rewrite (no rerank) ===
  s163 — Claims for rental bonds
  s186A — Regulations may establish rental bond roll-over scheme
  s156C — Termination notice for non-payment of rental bond
  s166 — Matters that may be subject of rental bond claim
  s174 — Repayment of bond to former co-tenant

=== Full pipeline (rewrite + hybrid + rerank) ===
  [+0.988] s163 — Claims for rental bonds
  [+0.978] s165 — Notice to tenants of claims against tenants
  [+0.958] s186A — Regulations may establish rental bond roll-over scheme
  [+0.957] s156D — Payment of rental bond during social housing tenancy agreements
  [+0.952] s166 — Matters that may be subject of rental bond claim


In [2]:
from dotenv import load_dotenv
load_dotenv()

from tenantmate.agent.graph import run_agent

# Pure retrieval question
result = run_agent("How much notice for a rent increase in NSW?")
print("ANSWER:\n", result["final_answer"])
print("\nChunks used:", [c["chunk_id"] for c in result["retrieved_chunks"]])
print("Tools used:", result["tool_results"])
print("Hops:", result["hop_count"])

/Users/varunchandrashekar/Tenantmate/code/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6254.81it/s]


ANSWER:
 # Rent Increase Notice Requirements in NSW

Under NSW rental law, a landlord must give **at least 60 days' written notice** before increasing rent.

## Key Requirements (Section 41):

1. **Notice Period**: The landlord must give written notice **at least 60 days before** the increased rent becomes payable

2. **Notice Content**: The written notice must specify:
   - The increased rent amount
   - The date from which it is payable

3. **Frequency Limits**: 
   - Rent cannot be increased within **12 months of the start of the tenancy**
   - Rent can only be increased **once per 12-month period**

## Additional Rights:

If you're in a fixed-term lease of 2 years or more and receive a rent increase notice, you have the right to terminate the tenancy without penalty under [Section 99]. You must give termination notice at least 21 days after receiving the rent increase notice, and before the increase takes effect.

This is general information, not legal advice.

Chunks used: ['NSW-R

In [2]:
from dotenv import load_dotenv
load_dotenv()

from tenantmate.agent.graph import run_agent

result = run_agent(
    "My tenancy started 1 March 2024. The landlord gave notice on "
    "1 May 2026 that rent goes from $600 to $680 from 1 June 2026. "
    "The last increase was 1 May 2025. Is this allowed?"
)

print(f"Hops: {result['hop_count']}")
print(f"\nTool results:")
for t in result['tool_results']:
    print(f"  {t}")
print(f"\nAnswer:\n{result['final_answer']}")

Hops: 3

Tool results:
  {'tool': 'rent_calculator', 'is_allowed': False, 'reasons': ['Notice given 31 days before the increase; 60 days required.'], 'earliest_lawful_date': datetime.date(2026, 6, 30), 'rule_citations': ['NSW RTA 2010 s 41(1)(b)', 'NSW RTA 2010 s 41(1A)(a)', 'NSW RTA 2010 s 41(1A)(b)']}

Answer:
# Analysis of Your Rent Increase

**No, this rent increase is not allowed.**

Your situation breaches two requirements under the Residential Tenancies Act 2010:

## 1. Insufficient Notice Period
The landlord gave notice on 1 May 2026 for a rent increase effective 1 June 2026. This is only **31 days' notice**, but [NSW-RTA2010-s41(1)(b)] requires **at least 60 days' notice** before the increased rent is payable.

## 2. Rent Increase Timing
The last rent increase occurred on 1 May 2025, and this increase is proposed for 1 June 2026. This is only **13 months apart**. However, [NSW-RTA2010-s41(1A)(b)] states rent "may not be increased more than once in any period of 12 months."

Si

In [3]:
from dotenv import load_dotenv
load_dotenv()

from tenantmate.agent.graph import run_agent


# ─── Case 1: SHOULD BE ALLOWED ──────────────────────────────────────
# - 90 days notice (≥ 60 ✓)
# - Tenancy started 27 months ago (≥ 12 ✓)
# - Last increase 14 months ago (≥ 12 ✓)
case_1 = (
    "My tenancy started on 1 January 2024. "
    "On 1 February 2026 the landlord gave written notice that rent "
    "will increase from $600 to $650 starting 1 May 2026. "
    "The last rent increase was on 1 December 2024. Is this allowed?"
)

# ─── Case 2: SHOULD FAIL on notice rule only ────────────────────────
# - 30 days notice (FAILS — needs 60)
# - Everything else fine
case_2 = (
    "My tenancy started on 1 January 2023. "
    "On 1 May 2026 the landlord gave written notice that rent "
    "will increase from $600 to $650 starting 1 June 2026. "
    "The last rent increase was on 1 January 2025. Is this allowed?"
)

# ─── Case 3: SHOULD FAIL on multiple rules ──────────────────────────
# - 31 days notice (FAILS notice)
# - Last increase 6 months ago (FAILS 12-month interval)
case_3 = (
    "My tenancy started on 1 January 2023. "
    "On 1 May 2026 the landlord gave written notice that rent "
    "will increase from $600 to $680 starting 1 June 2026. "
    "The last rent increase was on 1 December 2025. Is this allowed?"
)


for label, query in [("CASE 1 (allowed)", case_1),
                     ("CASE 2 (notice fail)", case_2),
                     ("CASE 3 (multiple fails)", case_3)]:
    print(f"\n{'='*70}\n{label}\n{'='*70}")
    result = run_agent(query)
    print(f"Hops: {result['hop_count']}")
    for t in result['tool_results']:
        print(f"Tool: {t}")
    print(f"\nAnswer:\n{result['final_answer'][:500]}...")


CASE 1 (allowed)
Hops: 3
Tool: {'tool': 'rent_calculator', 'is_allowed': True, 'reasons': [], 'earliest_lawful_date': None, 'rule_citations': ['NSW RTA 2010 s 41(1)(b)', 'NSW RTA 2010 s 41(1A)(a)', 'NSW RTA 2010 s 41(1A)(b)']}

Answer:
# Analysis of Your Rent Increase

Yes, this rent increase is **allowed**.

Here's why:

## 1. 60-Day Notice Requirement ✓
The landlord gave written notice on 1 February 2026 that the rent increase takes effect on 1 May 2026. This is 88 days' notice, which exceeds the minimum 60 days required under **s 41(1)(b)**.

## 2. 12-Month Restriction ✓
The relevant question is whether the increase complies with the restriction in **s 41(1A)(b)** — no more than one rent increase in any 12-month period.

- ...

CASE 2 (notice fail)
Hops: 3
Tool: {'tool': 'rent_calculator', 'is_allowed': False, 'reasons': ['Notice given 31 days before the increase; 60 days required.'], 'earliest_lawful_date': datetime.date(2026, 6, 30), 'rule_citations': ['NSW RTA 2010 s 41(1)(b)', 